# Notebook 02 — Parameter Estimation

**Member:** Nasya Putri Salsabila — Estimation Analyst (Member B)  
**Repository audited:** `pandas-dev/pandas`  

**Research questions addressed:**
- RQ1: What is the estimated probability that a pull request gets merged in pandas-dev/pandas?
- RQ2: Has the average issue closing time changed significantly after the last major release?

**Depends on:** `data/clean/dataset.csv` — output dari `01_eda.ipynb` (Member A)  
**Feeds into:** `03_confidence_interval.ipynb` (Member C) dan `04_hypothesis_testing.ipynb` (Member D)

## AI Usage Disclosure

**Member:** Nasya Putri Salsabila — Estimation Analyst | **Tools used:** Claude

| Task | Tool | Prompt summary | Output modified? |
|------|------|----------------|------------------|
| Scaffold `estimator.py` dan struktur notebook | Claude | "Scaffold MLE functions per Tsun 2020 for pandas-dev/pandas audit" | Ya — verifikasi semua formula, sesuaikan konteks pandas |

**Written entirely without AI:**
- Derivasi MLE Bernoulli (Section 1.1)
- Derivasi MLE Poisson (Section 2.1)
- Semua markdown interpretation cells (ditandai dengan ✍️)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys
import json
sys.path.insert(0, '..')   # agar src/ bisa dibaca dari folder notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import beta as beta_dist

from src.estimator import (
    mle_bernoulli,
    mle_poisson,
    beta_posterior,
    log_likelihood_bernoulli,
    log_likelihood_poisson,
)

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
print('Setup OK — semua imports berhasil.')

In [ ]:
# ── Load Dataset ────────────────────────────────────────────────────────────
df = pd.read_csv('../data/clean/dataset.csv', parse_dates=['created_at', 'closed_at'])

prs    = df[df['record_type'] == 'pr'].copy()
issues = df[df['record_type'] == 'issue'].copy()

print(f'Total records  : {len(df):,}')
print(f'PRs            : {len(prs):,}  (merged: {int(prs["merged_int"].sum())})')
print(f'Issues         : {len(issues):,}')
print(f'\nKolom dataset  : {list(df.columns)}')

---
## 1. Model Bernoulli — Probabilitas PR Di-merge

### Latar Belakang

Setiap PR yang ditutup di `pandas-dev/pandas` memiliki dua kemungkinan hasil:
- **X = 1** → PR di-merge (sukses)
- **X = 0** → PR ditutup tanpa di-merge (gagal)

Karena setiap PR adalah percobaan independen dengan dua hasil, kita modelkan dengan **Bernoulli(θ)**, di mana θ adalah probabilitas sebuah PR di-merge.

### 1.1 ✍️ Derivasi MLE Bernoulli — TULIS SENDIRI

> ⚠️ **Bagian ini WAJIB ditulis sendiri oleh Member B. Tidak boleh di-generate AI.**  
> Tuliskan langkah derivasi lengkap dari awal sampai dapat θ̂.

**Likelihood function:**
$$L(\theta) = \prod_{i=1}^{n} \theta^{x_i}(1-\theta)^{1-x_i} = \theta^k (1-\theta)^{n-k}$$

**Log-likelihood:**
$$\ln L(\theta) = \text{[TULIS SENDIRI]}$$

**Turunkan terhadap θ, samakan dengan nol:**
$$\frac{d \ln L}{d\theta} = \text{[TULIS SENDIRI]} = 0$$

**Solusi MLE:**
$$\hat{\theta}_{MLE} = \text{[TULIS SENDIRI]}$$

In [ ]:
# ── 1.2 Hitung MLE Bernoulli ────────────────────────────────────────────────
bernoulli_data = prs['merged_int'].dropna().astype(int).values
result_bern    = mle_bernoulli(bernoulli_data)

print('=== MLE Bernoulli — PR Merge Rate (pandas-dev/pandas) ===')
print(f"  k (merged PRs) : {result_bern['k']:,}")
print(f"  n (total PRs)  : {result_bern['n']:,}")
print(f"  θ̂  (MLE)       : {result_bern['theta_hat']:.4f}")
print(f"  SE(θ̂)          : {result_bern['se']:.4f}")

In [ ]:
# ── 1.3 Visualisasi Log-Likelihood Bernoulli ────────────────────────────────
k = result_bern['k']
n = result_bern['n']

theta_range = np.linspace(0.01, 0.99, 500)
ll_values   = [log_likelihood_bernoulli(t, k, n) for t in theta_range]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(theta_range, ll_values, color='steelblue', lw=2, label='ln L(θ)')
ax.axvline(result_bern['theta_hat'], color='tomato', ls='--', lw=1.5,
           label=f"θ̂ = {result_bern['theta_hat']:.4f}")
ax.set_xlabel('θ (probabilitas PR di-merge)')
ax.set_ylabel('Log-Likelihood  ln L(θ)')
ax.set_title('Bernoulli Log-Likelihood — pandas-dev/pandas PR Merge Rate')
ax.legend()
plt.tight_layout()
plt.savefig('../report/fig_ll_bernoulli.png', dpi=150)
plt.show()

### 1.4 ✍️ Interpretasi MLE Bernoulli — TULIS SENDIRI

> Jelaskan apa arti nilai θ̂ dalam konteks `pandas-dev/pandas`.  
> Apakah angka ini masuk akal untuk proyek open-source seperti pandas?  
> Apa implikasinya bagi maintainer?

*(tulis interpretasimu di sini)*

---
## 2. Model Poisson — Rata-rata Bug Report per Bulan

### Latar Belakang

Jumlah bug issues yang dibuka setiap bulan adalah **count data** (non-negatif, diskrit).  
Ini cocok dimodelkan dengan **Poisson(λ)**, di mana λ adalah rata-rata bug reports per bulan.

### 2.1 ✍️ Derivasi MLE Poisson — TULIS SENDIRI

> ⚠️ **Bagian ini WAJIB ditulis sendiri oleh Member B. Tidak boleh di-generate AI.**

**Likelihood function:**
$$L(\lambda) = \prod_{i=1}^{n} \frac{e^{-\lambda} \lambda^{x_i}}{x_i!}$$

**Log-likelihood:**
$$\ln L(\lambda) = \text{[TULIS SENDIRI]}$$

**Turunkan terhadap λ, samakan dengan nol:**
$$\frac{d \ln L}{d\lambda} = \text{[TULIS SENDIRI]} = 0$$

**Solusi MLE:**
$$\hat{\lambda}_{MLE} = \text{[TULIS SENDIRI]}$$

In [ ]:
# ── 2.2 Siapkan data: jumlah bug issues per bulan ──────────────────────────
bug_issues   = issues[issues['is_bug'] == 1].copy()
monthly_bugs = (
    bug_issues.groupby('year_month')
    .size()
    .reset_index(name='bug_count')
    .sort_values('year_month')
)

print(f'Bulan tercakup   : {len(monthly_bugs)}')
print(f'Total bug issues : {monthly_bugs["bug_count"].sum():,}')
print('\n5 bulan terakhir:')
print(monthly_bugs.tail(5).to_string(index=False))

In [ ]:
# ── 2.3 Hitung MLE Poisson ──────────────────────────────────────────────────
poisson_data = monthly_bugs['bug_count'].values
result_pois  = mle_poisson(poisson_data)

print('=== MLE Poisson — Monthly Bug Report Rate (pandas-dev/pandas) ===')
print(f"  n (bulan)      : {result_pois['n']}")
print(f"  λ̂  (MLE)       : {result_pois['lambda_hat']:.4f}")
print(f"  SE(λ̂)          : {result_pois['se']:.4f}")

In [ ]:
# ── 2.4 Visualisasi Log-Likelihood Poisson ─────────────────────────────────
lam_range = np.linspace(0.5, result_pois['lambda_hat'] * 2, 500)
ll_pois   = [log_likelihood_poisson(l, poisson_data) for l in lam_range]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lam_range, ll_pois, color='seagreen', lw=2, label='ln L(λ)')
ax.axvline(result_pois['lambda_hat'], color='tomato', ls='--', lw=1.5,
           label=f"λ̂ = {result_pois['lambda_hat']:.2f}")
ax.set_xlabel('λ (rata-rata bug reports per bulan)')
ax.set_ylabel('Log-Likelihood  ln L(λ)')
ax.set_title('Poisson Log-Likelihood — pandas-dev/pandas Monthly Bug Rate')
ax.legend()
plt.tight_layout()
plt.savefig('../report/fig_ll_poisson.png', dpi=150)
plt.show()

### 2.5 ✍️ Interpretasi MLE Poisson — TULIS SENDIRI

> Jelaskan arti λ̂ dalam konteks `pandas-dev/pandas`.  
> Apakah angka ini konsisten dengan ukuran dan aktivitas proyek?  
> Bagaimana tren bug report dari waktu ke waktu?

*(tulis interpretasimu di sini)*

---
## 3. Beta Posterior — Bayesian Update untuk PR Merge Rate

In [ ]:
# ── 3.1 Hitung Beta Posterior ───────────────────────────────────────────────
k = result_bern['k']
m = result_bern['n'] - result_bern['k']   # jumlah failures

result_beta = beta_posterior(k=k, m=m)

print('=== Beta Posterior — PR Merge Probability (pandas-dev/pandas) ===')
print(f"  k (merged)       : {k:,}")
print(f"  m (unmerged)     : {m:,}")
print(f"  α = k+1          : {result_beta['alpha']:,}")
print(f"  β = m+1          : {result_beta['beta']:,}")
print(f"  Posterior mode   : {result_beta['mode']:.4f}")
print(f"  Posterior mean   : {result_beta['mean']:.4f}")
print(f"  95% credible int.: {result_beta['ci_95']}")

In [ ]:
# ── 3.2 Visualisasi Beta Posterior ─────────────────────────────────────────
alpha = result_beta['alpha']
beta  = result_beta['beta']

# Zoom in karena n besar → posterior sangat tajam
lo = max(0.001, result_beta['mean'] - 0.05)
hi = min(0.999, result_beta['mean'] + 0.05)
theta_range = np.linspace(lo, hi, 500)
pdf_values  = beta_dist.pdf(theta_range, alpha, beta)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(theta_range, pdf_values, color='darkorchid', lw=2,
        label=f'Beta(α={alpha}, β={beta})')
ax.axvline(result_beta['mode'], color='tomato', ls='--', lw=1.5,
           label=f"Mode = {result_beta['mode']:.4f}")
ax.axvline(result_beta['ci_95'][0], color='gray', ls=':', lw=1)
ax.axvline(result_beta['ci_95'][1], color='gray', ls=':', lw=1,
           label=f"95% CI: {result_beta['ci_95']}")
ax.set_xlabel('θ (probabilitas PR di-merge)')
ax.set_ylabel('Density')
ax.set_title('Beta Posterior — PR Merge Probability (pandas-dev/pandas)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../report/fig_beta_posterior.png', dpi=150)
plt.show()

### 3.3 ✍️ Interpretasi Beta Posterior — TULIS SENDIRI

> Jelaskan perbedaan antara MLE θ̂ dan posterior mode.  
> Apa makna credible interval 95% ini?  
> Mengapa kita gunakan prior Beta(1,1)? Apa asumsinya?  
> **Hati-hati:** credible interval ≠ confidence interval — jelaskan perbedaannya.

*(tulis interpretasimu di sini)*

---
## 4. ✍️ Ringkasan & Handoff ke Layer Berikutnya — TULIS SENDIRI

> Rangkum semua hasil estimasi di bawah ini untuk digunakan Member C dan D.  
> Nilai-nilai ini yang akan di-pass ke notebook confidence interval dan hypothesis testing.

*(tulis ringkasanmu di sini)*

In [ ]:
# ── Ekspor hasil estimasi untuk Member C & D ────────────────────────────────
estimation_results = {
    'bernoulli': result_bern,
    'poisson'  : result_pois,
    'beta'     : {
        'alpha': result_beta['alpha'],
        'beta' : result_beta['beta'],
        'mode' : result_beta['mode'],
        'mean' : result_beta['mean'],
        'ci_95': list(result_beta['ci_95']),
    },
}

with open('../data/clean/estimation_results.json', 'w') as f:
    json.dump(estimation_results, f, indent=2)

print('estimation_results.json tersimpan di data/clean/')
print(json.dumps(estimation_results, indent=2))